# Dados em Painel

O [Módulo 07](../07_Econometria_I) tratou o dataset de companhias aéreas como uma
amostra *pooled* — e deixou uma pista sem resolver: o Durbin-Watson baixo do modelo
sugeria autocorrelação, e permitir um intercepto por empresa (dummies) melhorou bastante
o ajuste. Este módulo formaliza essa intuição: quando a **mesma unidade** (empresa,
cidade, país, indivíduo) é observada em **múltiplos períodos**, os dados têm estrutura de
**painel** — e ignorá-la joga fora informação, ou pior, vicia as estimativas.

Pré-requisito: [Módulo 07 — Econometria I](../07_Econometria_I)

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from linearmodels.panel import PooledOLS, PanelOLS, RandomEffects
from scipy import stats

pd.set_option('display.precision', 4)

## 1. Estrutura de dados em painel

Um painel combina a dimensão **cross-section** (unidades: $i = 1, \dots, N$) com a
dimensão **temporal** ($t = 1, \dots, T$). A vantagem central sobre um corte transversal
comum: se cada unidade tem características próprias que não mudam no tempo (cultura
organizacional de uma empresa, geografia de uma cidade) e que também afetam a variável
dependente, um painel permite **controlar essa heterogeneidade não-observada** — algo
impossível com uma única foto no tempo.

Retomamos o dataset de companhias aéreas do módulo anterior.

In [2]:
df_aereas = pd.read_csv('../07_Econometria_I/airline_costs.csv')
df_aereas = df_aereas.set_index(['I', 'T'])  # (entidade, tempo) -- exigido pelo linearmodels

y = df_aereas['ln_C']
X = sm.add_constant(df_aereas[['ln_Q', 'ln_PF', 'ln_LF']])

print(f"{df_aereas.index.get_level_values('I').nunique()} entidades x "
      f"{df_aereas.index.get_level_values('T').nunique()} períodos = {len(df_aereas)} observações")

6 entidades x 15 períodos = 90 observações


## 2. Pooled OLS

O ponto de partida — MQO comum, ignorando a repetição de unidades. É o mesmo modelo do
Módulo 07, agora rodado com `linearmodels` (que também serve de base de comparação para
os modelos seguintes).

In [3]:
pooled = PooledOLS(y, X).fit()
print(pooled.summary)

                          PooledOLS Estimation Summary                          
Dep. Variable:                   ln_C   R-squared:                        0.9883
Estimator:                  PooledOLS   R-squared (Between):              0.9869
No. Observations:                  90   R-squared (Within):               0.9909
Date:                Thu, Aug 06 2026   R-squared (Overall):              0.9883
Time:                        02:35:49   Log-likelihood                    61.624
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      2411.4
Entities:                           6   P-value                           0.0000
Avg Obs:                       15.000   Distribution:                    F(3,86)
Min Obs:                       15.000                                           
Max Obs:                       15.000   F-statistic (robust):             2411.4
                            

## 3. Modelo de Efeitos Fixos

Adiciona um intercepto próprio para cada entidade ($\alpha_i$), absorvendo toda
heterogeneidade **constante no tempo** — observada ou não:

$$Y_{it} = \alpha_i + \beta_1 X_{1it} + \dots + \beta_k X_{kit} + u_{it}$$

Isso só funciona porque $\alpha_i$ some ao considerar variações **dentro de cada
entidade** ao longo do tempo (*within estimator*) — por construção, qualquer variável que
não varia no tempo para uma mesma unidade (ex: a "cultura" de uma empresa) é
automaticamente controlada, mesmo sem ser medida.

In [4]:
fe = PanelOLS(y, X, entity_effects=True).fit()
print(fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                   ln_C   R-squared:                        0.9923
Estimator:                   PanelOLS   R-squared (Between):              0.9824
No. Observations:                  90   R-squared (Within):               0.9923
Date:                Thu, Aug 06 2026   R-squared (Overall):              0.9858
Time:                        02:35:49   Log-likelihood                    128.25
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      3460.0
Entities:                           6   P-value                           0.0000
Avg Obs:                       15.000   Distribution:                    F(3,81)
Min Obs:                       15.000                                           
Max Obs:                       15.000   F-statistic (robust):             3460.0
                            

Repare que os coeficientes de Efeitos Fixos são **idênticos** aos da regressão com
dummies por empresa do Módulo 07 — não é coincidência: regressão com dummy de entidade
(LSDV, *Least Squares Dummy Variable*) e Efeitos Fixos são **algebricamente o mesmo
estimador**; `linearmodels` só calcula de um jeito mais eficiente (sem estimar
explicitamente cada $\alpha_i$).

## 4. Modelo de Efeitos Aleatórios

Também modela heterogeneidade entre entidades, mas trata $\alpha_i$ como uma variável
**aleatória** (não um parâmetro fixo a estimar), assumindo que ela **não é correlacionada**
com as variáveis explicativas. Quando esse pressuposto é razoável, Efeitos Aleatórios é
mais eficiente que Efeitos Fixos (usa também a variação **entre** entidades, não só
dentro delas) — mas se a heterogeneidade estiver correlacionada com os regressores, o
estimador fica viesado.

In [5]:
re = RandomEffects(y, X).fit()
print(re.summary)

                        RandomEffects Estimation Summary                        
Dep. Variable:                   ln_C   R-squared:                        0.9920
Estimator:              RandomEffects   R-squared (Between):              0.9839
No. Observations:                  90   R-squared (Within):               0.9922
Date:                Thu, Aug 06 2026   R-squared (Overall):              0.9868
Time:                        02:35:49   Log-likelihood                    125.35
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      3549.3
Entities:                           6   P-value                           0.0000
Avg Obs:                       15.000   Distribution:                    F(3,86)
Min Obs:                       15.000                                           
Max Obs:                       15.000   F-statistic (robust):             3549.3
                            

## 5. Teste de Hausman: qual modelo usar?

$H_0$: a heterogeneidade não-observada **não é correlacionada** com os regressores (RE é
consistente e mais eficiente — prefira RE). $H_1$: há correlação (RE é viesado — só FE é
consistente).

$$H = (\hat{\beta}_{FE} - \hat{\beta}_{RE})' \left[Var(\hat{\beta}_{FE}) - Var(\hat{\beta}_{RE})\right]^{-1} (\hat{\beta}_{FE} - \hat{\beta}_{RE}) \sim \chi^2_k$$

O `linearmodels` não traz um `.hausman()` pronto — calculamos manualmente pela fórmula
(comum na prática, é só álgebra matricial):

In [6]:
def teste_hausman(fe_result, re_result):
    variaveis_comuns = [v for v in fe_result.params.index if v != 'const']

    b_fe = fe_result.params[variaveis_comuns]
    b_re = re_result.params[variaveis_comuns]
    v_fe = fe_result.cov.loc[variaveis_comuns, variaveis_comuns]
    v_re = re_result.cov.loc[variaveis_comuns, variaveis_comuns]

    diferenca = b_fe - b_re
    cov_diferenca = v_fe - v_re

    estatistica = diferenca.values @ np.linalg.inv(cov_diferenca.values) @ diferenca.values
    graus_liberdade = len(variaveis_comuns)
    p_valor = 1 - stats.chi2.cdf(estatistica, graus_liberdade)

    return estatistica, graus_liberdade, p_valor

h_stat, h_df, h_pvalor = teste_hausman(fe, re)
print(f"Estatística de Hausman = {h_stat:.4f}")
print(f"Graus de liberdade      = {h_df}")
print(f"p-valor                 = {h_pvalor:.4f}")

if h_pvalor < 0.05:
    print("\np-valor < 5%: rejeitamos H0 — a heterogeneidade parece correlacionada com")
    print("os regressores. Use Efeitos Fixos.")
else:
    print("\np-valor > 5%: não rejeitamos H0 — não há evidência forte de que RE seja")
    print("inconsistente aqui. RE seria preferível por ser mais eficiente, embora a")
    print("diferença de coeficientes entre FE e RE já pareça pequena na prática.")

Estatística de Hausman = 4.5745
Graus de liberdade      = 3
p-valor                 = 0.2057

p-valor > 5%: não rejeitamos H0 — não há evidência forte de que RE seja
inconsistente aqui. RE seria preferível por ser mais eficiente, embora a
diferença de coeficientes entre FE e RE já pareça pequena na prática.


## 6. Estudo de caso — quando ignorar o painel distorce a conclusão

Nem sempre a diferença entre Pooled OLS e Efeitos Fixos é pequena. Um exemplo clássico da
literatura (Wooldridge): dados de 64 cidades universitárias americanas, em 1980 e 1990,
relacionando o **aluguel médio** à população, à renda per capita e ao **percentual de
estudantes na população** (`pctstu`) — a pergunta de política pública é se a presença de
estudantes pressiona o preço dos aluguéis.

In [7]:
df_aluguel = pd.read_csv('rental_housing.csv')
df_aluguel = df_aluguel.set_index(['city', 'year'])

y2 = df_aluguel['l_rent']  # log do aluguel médio
X2 = sm.add_constant(df_aluguel[['l_pop', 'l_avginc', 'pctstu']])

pooled2 = PooledOLS(y2, X2).fit()
fe2 = PanelOLS(y2, X2, entity_effects=True).fit()

print(f"Efeito de pctstu (Pooled OLS): {pooled2.params['pctstu']:.4f}  (p={pooled2.pvalues['pctstu']:.4f})")
print(f"Efeito de pctstu (Efeitos Fixos): {fe2.params['pctstu']:.4f}  (p={fe2.pvalues['pctstu']:.4f})")

Efeito de pctstu (Pooled OLS): 0.0066  (p=0.0000)
Efeito de pctstu (Efeitos Fixos): 0.0187  (p=0.0077)


O efeito estimado por Efeitos Fixos é quase **3× maior** que o do Pooled OLS. Faz
sentido: cidades com mais estudantes tendem a ter outras características de tempo
constante (proximidade de centros urbanos, atratividade) que também afetam o aluguel —
Pooled OLS confunde esse efeito com o de `pctstu`. Efeitos Fixos isola apenas a variação
**dentro de cada cidade** entre 1980 e 1990, purgando esse viés de variável omitida.

O teste F de "poolability" do próprio `linearmodels` confirma que ignorar a estrutura de
painel não é uma boa ideia aqui:

In [8]:
print(fe2.f_pooled)  # F-test for Poolability: H0 = os efeitos de entidade são todos zero

Pooled F-statistic
H0: Effects are zero
Statistic: 3.0411
P-value: 0.0000
Distributed: F(63,61)


## 7. Ponte para séries temporais

Um painel é, estruturalmente, uma coleção de $N$ séries temporais curtas (uma por
entidade). No limite em que $N=1$ — uma única entidade observada por muitos períodos —,
a estrutura de painel colapsa para uma **série temporal pura**, o assunto do
[Módulo 09](../09_Séries_Temporais). Os dois módulos de Econometria compartilham essa
ideia de dependência ao longo do tempo; a diferença é que painel tem a dimensão extra de
múltiplas unidades para ajudar a isolar heterogeneidade não-observada — recurso que uma
série temporal isolada não tem.

## 8. Exercícios de fixação

1. Rode um modelo Pooled OLS para o dataset de aluguel (`df_aluguel`) e compare o R²
   com o do modelo de Efeitos Fixos (`fe2`) — qual é maior, e por quê?
2. Calcule o teste de Hausman entre Efeitos Fixos e Efeitos Aleatórios para o dataset de
   aluguel. O resultado aponta para qual modelo?

In [9]:
# 1.
print(f"1) R² Pooled: {pooled2.rsquared:.4f}   R² Efeitos Fixos: {fe2.rsquared:.4f}")
print("   Aqui o R² do FE é MAIOR: as 3 variáveis explicam bem a variação dentro de cada")
print("   cidade entre 1980 e 1990, mas deixam de fora bastante heterogeneidade ENTRE")
print("   cidades (renda/atratividade não captadas pelo modelo) -- por isso o Pooled,")
print("   que precisa explicar as duas fontes de variação de uma vez, fica com R² menor.")
print("   Não existe uma regra geral de qual R² é maior -- depende de quanto de cada tipo")
print("   de variação (dentro x entre) os regressores conseguem captar.")

# 2.
re2 = RandomEffects(y2, X2).fit()
h_stat2, h_df2, h_pvalor2 = teste_hausman(fe2, re2)
print(f"\n2) Hausman: estatística={h_stat2:.4f}  p-valor={h_pvalor2:.4f}")
print("   Rejeita H0 -> Efeitos Fixos é a escolha adequada aqui."
      if h_pvalor2 < 0.05 else
      "   Não rejeita H0 -> Efeitos Aleatórios seria defensável aqui.")

1) R² Pooled: 0.7971   R² Efeitos Fixos: 0.9337
   Aqui o R² do FE é MAIOR: as 3 variáveis explicam bem a variação dentro de cada
   cidade entre 1980 e 1990, mas deixam de fora bastante heterogeneidade ENTRE
   cidades (renda/atratividade não captadas pelo modelo) -- por isso o Pooled,
   que precisa explicar as duas fontes de variação de uma vez, fica com R² menor.
   Não existe uma regra geral de qual R² é maior -- depende de quanto de cada tipo
   de variação (dentro x entre) os regressores conseguem captar.

2) Hausman: estatística=258.8498  p-valor=0.0000
   Rejeita H0 -> Efeitos Fixos é a escolha adequada aqui.


## Encerramento do módulo

Com Pooled OLS, Efeitos Fixos, Efeitos Aleatórios e o teste de Hausman, o Módulo 08 está
completo — junto com o Módulo 07, fecha o bloco de Econometria "clássica". O
[Módulo 09 — Séries Temporais](../09_Séries_Temporais) já existe no curso (previsão de
IPCA e SELIC com ARIMA) e agora ganha reforço teórico do que os módulos de Econometria
construíram sobre dependência temporal.

## Bibliografia
- WOOLDRIDGE, Jeffrey M. *Introdução à Econometria: uma abordagem moderna*. São Paulo: Cengage Learning, 2016. (fonte do dataset de aluguel/cidades universitárias)
- WOOLDRIDGE, Jeffrey M. *Econometric Analysis of Cross Section and Panel Data*. 2. ed. MIT Press, 2010.
- BALTAGI, Badi H. *Econometric Analysis of Panel Data*. 6. ed. Springer, 2021.
- GREENE, William H. *Econometric Analysis*. 8. ed. Pearson, 2018.